# Notebook 01 — Exploratory Data Analysis: OhioT1DM

**Proyek:** Prediction-Conditioned RAG untuk Pengelolaan Diabetes
**Dataset:** OhioT1DM (Marling & Bunescu, 2020) — 12 pasien Diabetes Tipe 1, sinyal CGM 5-menit.

## Tujuan Notebook
Notebook ini melakukan **eksplorasi data (EDA)** terhadap dataset OhioT1DM yang sudah
diparse ke `data/raw/ohio_t1dm_merged.csv`. Fokusnya adalah *memahami karakteristik data*
sebelum pemodelan — **bukan** melatih model (pelatihan dilakukan via `src/models/` dan
`scripts/`, bukan di notebook).

Yang akan dijawab notebook ini:
1. Seberapa bersih dan lengkap datanya? (kualitas data)
2. Bagaimana distribusi glukosa relatif terhadap zona klinis (hipo/target/hiper)?
3. Apa pola temporal glukosa (diurnal)?
4. Seberapa jarang (*sparse*) kanal event seperti karbohidrat/insulin/aktivitas?
5. Bagaimana hubungan antar-fitur?
6. Bagaimana perbandingan cadence **CGM (5-menit)** vs **SMBG (surrogate)** — landasan
   metodologi (lihat `docs/METHODOLOGY.md`).

> Semua figur disimpan ke `results/eda/` agar konsisten dengan artefak evaluasi lain.

## 1. Setup & Pemuatan Data

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Agar bisa import modul src/ saat dijalankan dari folder notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# Folder output figur EDA
EDA_OUT = PROJECT_ROOT / "results" / "eda"
EDA_OUT.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ohio_t1dm_merged.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values(["patient_id", "timestamp"]).reset_index(drop=True)
print(f"Dataset dimuat: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.head()

: 

## 2. Kualitas Data

Sebelum analisis lebih jauh, kita periksa integritas dasar: jumlah pasien, rentang waktu,
tipe data, dan nilai hilang. Data CGM klinis sering punya *gap*; OhioT1DM yang sudah diparse
diharapkan sudah rapi.

In [ ]:
print("=== Ringkasan Struktur ===")
print(f"Jumlah baris      : {len(df):,}")
print(f"Jumlah pasien     : {df['patient_id'].nunique()}")
print(f"Pasien            : {sorted(df['patient_id'].unique())}")
print(f"Rentang waktu     : {df['timestamp'].min()}  ->  {df['timestamp'].max()}")
print(f"Total nilai hilang: {int(df.isnull().sum().sum())}")
print()
print("=== Tipe Data per Kolom ===")
print(df.dtypes)

In [ ]:
# Cadence: selisih antar pembacaan per pasien (verifikasi CGM 5-menit)
gaps = df.groupby("patient_id")["timestamp"].diff().dropna().dt.total_seconds() / 60
print(f"Median gap antar pembacaan : {gaps.median():.1f} menit  (CGM ~5 menit)")
print(f"Persentil 90 gap           : {gaps.quantile(0.90):.1f} menit")
print(f"Gap maksimum               : {gaps.max():.1f} menit  (indikasi sensor lepas/jeda)")

**Interpretasi (isi sesuai hasil run):**
Dataset bersih (0 nilai hilang) dengan cadence median 5 menit, mengonfirmasi sumber CGM.
Gap besar sesekali wajar pada data CGM nyata (sensor dilepas/diganti). _Tuliskan angka aktual
dari sel di atas ke laporan Bab III._

## 3. Distribusi Glukosa & Zona Klinis

Glukosa darah dibagi menjadi tiga zona klinis standar (ADA 2023):
- **Hipoglikemia**: < 70 mg/dL (berbahaya, butuh tindakan cepat)
- **Target**: 70–180 mg/dL (rentang aman)
- **Hiperglikemia**: > 180 mg/dL (perlu pemantauan/koreksi)

Distribusi ini penting karena sistem kita memprediksi ke zona mana glukosa akan bergerak,
lalu mengondisikan retrieval RAG pada prediksi tersebut.

In [ ]:
LOW, HIGH = 70, 180
pct_hypo = (df["glucose"] < LOW).mean() * 100
pct_target = ((df["glucose"] >= LOW) & (df["glucose"] <= HIGH)).mean() * 100
pct_hyper = (df["glucose"] > HIGH).mean() * 100
print(f"Hipoglikemia (<70)   : {pct_hypo:5.2f}%")
print(f"Target (70-180)      : {pct_target:5.2f}%")
print(f"Hiperglikemia (>180) : {pct_hyper:5.2f}%")
print(f"Glukosa rata-rata    : {df['glucose'].mean():.1f} mg/dL (std {df['glucose'].std():.1f})")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df["glucose"], bins=80, color="#4c72b0", alpha=0.85)
ax.axvspan(0, LOW, color="#e74c3c", alpha=0.12, label="Hipoglikemia <70")
ax.axvspan(LOW, HIGH, color="#2ecc71", alpha=0.12, label="Target 70-180")
ax.axvspan(HIGH, 400, color="#f39c12", alpha=0.12, label="Hiperglikemia >180")
ax.set_xlabel("Glukosa (mg/dL)")
ax.set_ylabel("Frekuensi")
ax.set_title("Distribusi Glukosa dengan Zona Klinis")
ax.legend()
fig.tight_layout()
fig.savefig(EDA_OUT / "01_glucose_distribution.png", dpi=150)
plt.show()

## 4. Pola Temporal (Diurnal)

Glukosa memiliki ritme harian (dawn phenomenon, lonjakan postprandial). Kita lihat rata-rata
glukosa per jam dalam sehari untuk menangkap pola yang dapat dimanfaatkan model.

In [ ]:
df["hour"] = df["timestamp"].dt.hour
hourly = df.groupby("hour")["glucose"].agg(["mean", "std"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hourly.index, hourly["mean"], marker="o", color="#4c72b0")
ax.fill_between(hourly.index, hourly["mean"] - hourly["std"],
                hourly["mean"] + hourly["std"], alpha=0.15, color="#4c72b0")
ax.axhspan(70, 180, color="#2ecc71", alpha=0.08)
ax.set_xlabel("Jam (0-23)")
ax.set_ylabel("Glukosa rata-rata (mg/dL)")
ax.set_title("Pola Diurnal Glukosa (mean +/- std)")
ax.set_xticks(range(0, 24, 2))
fig.tight_layout()
fig.savefig(EDA_OUT / "02_diurnal_pattern.png", dpi=150)
plt.show()

In [ ]:
# Contoh deret waktu 1 pasien selama ~2 hari pertama
pid = sorted(df["patient_id"].unique())[0]
sample = df[df["patient_id"] == pid].head(576)  # 576 x 5min = 48 jam

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sample["timestamp"], sample["glucose"], color="#4c72b0", lw=1)
ax.axhspan(70, 180, color="#2ecc71", alpha=0.10)
ax.set_title(f"Deret Waktu Glukosa — {pid} (48 jam pertama)")
ax.set_xlabel("Waktu")
ax.set_ylabel("Glukosa (mg/dL)")
fig.tight_layout()
fig.savefig(EDA_OUT / "03_sample_timeseries.png", dpi=150)
plt.show()

## 5. Sparsity Kanal Event (Karbohidrat, Insulin, Aktivitas)

Selain glukosa (yang padat 5-menit), kanal *event* seperti karbohidrat, insulin, dan
aktivitas hanya tercatat saat kejadian terjadi. Karena event tidak selalu jatuh tepat pada
timestamp CGM, kanal-kanal ini **sangat jarang bernilai non-nol**. Ini temuan penting:
model harus belajar dari sinyal yang didominasi nol pada kanal-kanal tersebut.

In [ ]:
event_cols = ["carbs", "insulin", "activity", "stress", "sleep", "work", "illness"]
rows = []
for c in event_cols:
    nz = (df[c] > 0).mean() * 100
    rows.append({"fitur": c, "% non-nol": round(nz, 3), "maks": df[c].max(), "mean": round(df[c].mean(), 3)})
sparsity = pd.DataFrame(rows)
print(sparsity.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(sparsity["fitur"], sparsity["% non-nol"], color="#c44e52", alpha=0.85)
ax.set_ylabel("% baris dengan nilai > 0")
ax.set_title("Sparsity Kanal Event (semakin rendah = semakin jarang tercatat)")
for i, v in enumerate(sparsity["% non-nol"]):
    ax.text(i, v, f"{v:.2f}%", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig(EDA_OUT / "04_event_sparsity.png", dpi=150)
plt.show()

**Interpretasi (TEMUAN PENTING):** Pemeriksaan mengungkap bahwa beberapa kanal *degenerate*
pada resolusi 5-menit:
- **`insulin` praktis nol total** — event insulin (bolus/basal) hampir tidak pernah jatuh
  tepat pada timestamp grid CGM 5-menit, sehingga setelah agregasi kolom ini ~selalu 0.
- **`stress` konstan (nilai default 5)** — tidak membawa variasi/informasi.
- **`sleep`, `work`, `illness` nol**; `activity` & `carbs` sangat jarang (<0.3%).

Akibatnya, dari 5 fitur model `[glucose, carbs, insulin, activity, stress]`, secara efektif
**hanya glukosa historis** (dan sedikit `carbs`) yang membawa sinyal prediktif. Implikasi:
- Prediksi pada dasarnya memanfaatkan **autokorelasi glukosa**, bukan multimodalitas penuh.
- Tetap menjadi justifikasi **Random Forest** (robust terhadap fitur konstan/sparse).
- Klaim "multimodal" harus diperhalus di laporan; alignment event→grid adalah keterbatasan
  parser/data yang perlu disebutkan.
- _Feeds: Bab III (analisis data), Bab VI (limitasi: fitur event degenerate & alignment)._

## 6. Statistik Ringkas & Distribusi Fitur Numerik

In [ ]:
numeric_cols = ["glucose", "carbs", "insulin", "activity", "stress"]
display_stats = df[numeric_cols].describe().T
print(display_stats.to_string())

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, numeric_cols):
    ax.hist(df[col], bins=50, color="#4c72b0", alpha=0.8)
    ax.set_title(col)
    ax.set_yscale("log")  # skala log agar ekor sparse terlihat
axes.flat[-1].axis("off")
fig.suptitle("Distribusi Fitur Numerik (sumbu-y log)")
fig.tight_layout()
fig.savefig(EDA_OUT / "05_feature_distributions.png", dpi=150)
plt.show()

## 7. Korelasi Antar-Fitur

Kita tambahkan fitur turunan `glucose_change` (selisih glukosa antar pembacaan, **per pasien**
agar tidak bocor antar-pasien) untuk melihat hubungan laju perubahan glukosa dengan fitur lain.

In [ ]:
df["glucose_change"] = df.groupby("patient_id")["glucose"].diff()

corr_cols = ["glucose", "glucose_change", "carbs", "insulin", "activity", "stress"]
corr = df[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Matriks Korelasi Antar-Fitur")
fig.tight_layout()
fig.savefig(EDA_OUT / "06_correlation_heatmap.png", dpi=150)
plt.show()

## 8. Variabilitas Antar-Pasien

Setiap pasien punya profil metabolik berbeda. Boxplot di bawah menunjukkan sebaran glukosa
per pasien — penting untuk justifikasi *split per pasien* (mencegah kebocoran data temporal).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
order = sorted(df["patient_id"].unique())
sns.boxplot(data=df, x="patient_id", y="glucose", order=order, ax=ax, color="#4c72b0", fliersize=1)
ax.axhspan(70, 180, color="#2ecc71", alpha=0.08)
ax.set_title("Distribusi Glukosa per Pasien")
ax.set_xlabel("Pasien")
ax.set_ylabel("Glukosa (mg/dL)")
plt.xticks(rotation=45)
fig.tight_layout()
fig.savefig(EDA_OUT / "07_per_patient_boxplot.png", dpi=150)
plt.show()

## 9. Analisis Cadence: CGM (5-menit) vs SMBG (Surrogate)

Sistem ini ditujukan untuk skenario **SMBG** (pengukuran mandiri 3–6x/hari), sementara
OhioT1DM adalah **CGM** padat (5-menit). Untuk menjembatani, `src/data/preprocessor.py`
menyediakan `downsample_smbg()`. Di sini kita visualisasikan perbedaan kedua cadence pada
satu pasien dan bandingkan secara kuantitatif.

> Detail justifikasi: lihat `docs/METHODOLOGY.md`.

In [ ]:
from src.data.preprocessor import DataPreprocessor

prep = DataPreprocessor({})
df_smbg = prep.downsample_smbg(df.drop(columns=["hour", "glucose_change"]), interval_minutes=240)

# Bandingkan cadence via MEDIAN GAP antar pembacaan (robust terhadap rentang tanggal
# OhioT1DM yang ter-anonimisasi & punya gap besar — sehingga "pembacaan/hari" menyesatkan).
def median_gap_min(d):
    g = d.sort_values(["patient_id", "timestamp"]).groupby("patient_id")["timestamp"].diff().dropna()
    return round(g.dt.total_seconds().median() / 60, 1)

cmp = pd.DataFrame({
    "metrik": ["total baris", "median gap (menit)", "glukosa mean", "glukosa std"],
    "CGM (5-min)": [len(df), median_gap_min(df), round(df["glucose"].mean(), 1), round(df["glucose"].std(), 1)],
    "SMBG (240-min)": [len(df_smbg), median_gap_min(df_smbg), round(df_smbg["glucose"].mean(), 1), round(df_smbg["glucose"].std(), 1)],
})
print(cmp.to_string(index=False))
print()
print(f"Rasio downsampling: {len(df) / len(df_smbg):.1f}x")

In [ ]:
# Overlay 1 pasien: CGM penuh vs titik SMBG
pid = sorted(df["patient_id"].unique())[0]
cgm_p = df[df["patient_id"] == pid].head(576)
smbg_p = df_smbg[df_smbg["patient_id"] == pid]
smbg_p = smbg_p[smbg_p["timestamp"].isin(cgm_p["timestamp"])]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(cgm_p["timestamp"], cgm_p["glucose"], color="#4c72b0", lw=1, alpha=0.7, label="CGM (5-menit)")
ax.scatter(smbg_p["timestamp"], smbg_p["glucose"], color="#c44e52", s=40, zorder=5, label="SMBG (240-menit)")
ax.axhspan(70, 180, color="#2ecc71", alpha=0.08)
ax.set_title(f"CGM vs SMBG Surrogate — {pid} (48 jam pertama)")
ax.set_xlabel("Waktu")
ax.set_ylabel("Glukosa (mg/dL)")
ax.legend()
fig.tight_layout()
fig.savefig(EDA_OUT / "08_cgm_vs_smbg.png", dpi=150)
plt.show()

**Interpretasi:** Downsampling SMBG mempertahankan rentang nilai glukosa namun kehilangan
detail tren intra-hari. Ini menggambarkan trade-off klinis nyata: SMBG murah & praktis tapi
miskin resolusi. Model utama dilatih pada CGM (sinyal kaya), sementara mode SMBG tersedia
untuk eksperimen robustness. _Feeds: Bab III metodologi, Bab VI limitasi surrogate._

## 10. Ringkasan Temuan Utama

Isi angka aktual setelah menjalankan notebook:

| # | Temuan | Implikasi untuk Laporan |
|---|--------|--------------------------|
| 1 | Data bersih (0 NaN), 12 pasien, cadence 5-menit | Bab III: kualitas & sumber data |
| 2 | ~__% waktu di zona target, ~__% hiper, ~__% hipo | Bab III/VI: karakteristik klinis |
| 3 | Pola diurnal terlihat (dawn phenomenon) | Bab III: justifikasi fitur temporal |
| 4 | Fitur degenerate: insulin=0 total, stress konstan, hanya glukosa bersinyal | Bab III/VI: justifikasi RF, limitasi multimodalitas |
| 5 | Variabilitas antar-pasien tinggi | Bab III: justifikasi split per pasien |
| 6 | SMBG surrogate kehilangan resolusi intra-hari | Bab III/VI: trade-off cadence |

> **Catatan reproduksi:** Notebook ini hanya EDA. Pelatihan model dilakukan via
> `python -m src.models.rf_model` dan `scripts/eval_rf_lstm.py`. Figur tersimpan di
> `results/eda/`.